# Os passes de cada seleção em 2018 e 2022

## Task 3: calcular os passes de cada seleção nas Copas de 2018 e 2022 e a média por partida

## O problema

Só as Copas de 2018 e 2022, as que têm todos os passes no feed. Duas etapas: 
- os passes de cada seleção **em cada partida**;
- depois a soma **por Copa**, com o número de partidas e a média.

In [6]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-shuffle")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o código dos vídeos 4.1 e 4.2 (lances, partidas e o ano em cada lance).

In [7]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.2: o ano em cada lance
lances = eventos.join(partidas.select("match_id", "ano"), on="match_id")

print(eventos.count())
print(partidas.count())
print(lances.count())

195747
964
195747


## Passes de cada seleção em cada partida

In [8]:
passes_por_partida = (
    lances
    .filter(F.col("ano").isin(2018, 2022))
    .filter(F.col("event_type") == "Pass")
    .groupBy("ano", "match_id", "team")
    .agg(F.count("*").alias("passes"))
)

In [9]:
passes_por_partida.orderBy("match_id").show(4)

+----+--------+------------+------+
| ano|match_id|        team|passes|
+----+--------+------------+------+
|2018|    1069|Saudi Arabia|   516|
|2018|    1069|      Russia|   311|
|2018|    1070|       Egypt|   421|
|2018|    1070|     Uruguay|   579|
+----+--------+------------+------+
only showing top 4 rows


## O que é um shuffle

Lembra das duas fases da aula 1 (somar cada pedaço e juntar os parciais)? Juntar é o **shuffle**: as linhas de cada
grupo precisam se encontrar numa partição.

```
    partição 1                         partição 2                         partição 3
    (2018, 1069, Russia):       150    (2018, 1069, Russia):       161    (2018, 1069, Saudi Arabia): 256
    (2018, 1069, Saudi Arabia): 260    (2018, 1070, Uruguay):      300    (2018, 1070, Uruguay):      279
               \                                  |                                  /
                shuffle: todas as linhas de um mesmo grupo vão para a mesma partição
                                                  |
                         (2018, 1069, Russia):       150 + 161 = 311
                         (2018, 1069, Saudi Arabia): 260 + 256 = 516
                         (2018, 1070, Uruguay):      300 + 279 = 579
```

Os totais são os da saída acima; a divisão entre as partições é só um exemplo.

In [10]:
passes_por_partida.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[ano#308, match_id#286, team#283], functions=[count(1)])
   +- Exchange hashpartitioning(ano#308, match_id#286, team#283, 200), ENSURE_REQUIREMENTS, [plan_id=1324]
      +- HashAggregate(keys=[ano#308, match_id#286, team#283], functions=[partial_count(1)])
         +- Project [match_id#286, team#283, ano#308]
            +- BroadcastHashJoin [match_id#286], [match_id#309], Inner, BuildRight, false, false
               :- Project [MatchID#279 AS match_id#286, team#283]
               :  +- Filter ((isnotnull(event_type#282) AND (event_type#282 = Pass)) AND isnotnull(MatchID#279))
               :     +- FileScan parquet [MatchID#279,event_type#282,team#283] Batched: true, DataFilters: [isnotnull(event_type#282), (event_type#282 = Pass), isnotnull(MatchID#279)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/events], PartitionFilters: [], PushedFilters: [IsNotNull(event_type)

Olhe só as três primeiras linhas abaixo do `AdaptiveSparkPlan`; o resto é o join do vídeo 4.2.

| No plano | O que faz |
|---|---|
| `HashAggregate(... partial_count(1))` | cada partição conta o que tem |
| `Exchange hashpartitioning(..., 200)` | o shuffle: cada grupo vai para uma partição |
| `HashAggregate(... count(1))` | soma os parciais de cada grupo |

`HashAggregate` é o `groupBy` por dentro: um caderninho com uma linha por grupo.

| Tipo | Precisa de dado de outras partições? | Exemplos |
|---|---|---|
| **estreita** (*narrow*) | não | `filter`, `select`, `withColumn` |
| **larga** (*wide*) | sim, exige shuffle | `groupBy`, `join` (sem broadcast), `distinct`, `orderBy`, `dropDuplicates` |

Cada shuffle começa um **stage** (etapa) novo.

## Agora por Copa

In [11]:
passes_por_copa = (
    passes_por_partida
    .groupBy("ano", F.col("team").alias("selecao"))
    .agg(
        F.sum("passes").alias("passes"),
        F.count("*").alias("partidas"),
        F.round(F.avg("passes"), 1).alias("media_por_partida"),
    )
)

In [12]:
passes_por_copa.orderBy("ano", F.desc("passes")).show(5)

+----+-------+------+--------+-----------------+
| ano|selecao|passes|partidas|media_por_partida|
+----+-------+------+--------+-----------------+
|2018|Croatia|  3708|       7|            529.7|
|2018|England|  3631|       7|            518.7|
|2018|Belgium|  3492|       7|            498.9|
|2018|  Spain|  3291|       4|            822.8|
|2018| France|  2931|       7|            418.7|
+----+-------+------+--------+-----------------+
only showing top 5 rows


## Quantos shuffles?

In [13]:
passes_por_copa.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[ano#308, team#283], functions=[sum(passes#437L), count(1), avg(passes#437L)])
   +- Exchange hashpartitioning(ano#308, team#283, 200), ENSURE_REQUIREMENTS, [plan_id=1750]
      +- HashAggregate(keys=[ano#308, team#283], functions=[partial_sum(passes#437L), partial_count(1), partial_avg(passes#437L)])
         +- HashAggregate(keys=[ano#308, match_id#286, team#283], functions=[count(1)])
            +- Exchange hashpartitioning(ano#308, match_id#286, team#283, 200), ENSURE_REQUIREMENTS, [plan_id=1746]
               +- HashAggregate(keys=[ano#308, match_id#286, team#283], functions=[partial_count(1)])
                  +- Project [match_id#286, team#283, ano#308]
                     +- BroadcastHashJoin [match_id#286], [match_id#309], Inner, BuildRight, false, false
                        :- Project [MatchID#279 AS match_id#286, team#283]
                        :  +- Filter ((isnotnull(event_type#282) AND 

Quantos `Exchange` o plano tem? De qual linha do código vem cada um?

## O número 200

`spark.sql.shuffle.partitions`: o número de partições depois de um shuffle (padrão: **200**). O **AQE** (Adaptive
Query Execution, o `AdaptiveSparkPlan` do topo do plano) olha o tamanho real do dado durante a execução e **junta as
partições pequenas**.

In [14]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

Na Spark UI, aba **SQL / DataFrame**, o ajuste do AQE aparece como **`AQEShuffleRead`**.

## Para consultar depois

- **`repartition(n)`** redistribui o dado em `n` partições **com shuffle**, equilibradas; **`coalesce(n)`** só junta
  partições vizinhas, **sem shuffle**, e só diminui o número.
- **`orderBy`** também é uma transformação larga: para ordenar o dado inteiro, cada faixa de valores vai para uma
  partição (`rangepartitioning` no plano).
- **Mudando o padrão:** `spark.conf.set("spark.sql.shuffle.partitions", "8")`. Com o AQE ligado, raramente é
  necessário.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [15]:
spark.stop()